# GOV-01 Model Gate: First CNN Baseline

This notebook trains the first simple image model for GOV-01. It predicts only `Normal` or `Pothole`.

**Important:** it uses training and validation images only. The clean `test` folder stays untouched until a final model is selected.

## Before running in Google Colab

1. Clone this repository in Colab.
2. Upload and extract the original dataset ZIP.
3. Run `src/build_clean_split.py` with seed `42`.
4. Confirm `data/processed/clean_split/train`, `validation`, and `test` exist.
5. Run the cells below in order.

Do not open, score, or tune against the clean `test` split in this notebook.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score, precision_score, recall_score, roc_auc_score

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_NAMES = ['Normal', 'Pothole']
DATA_DIR = Path('data/processed/clean_split')
REPORTS_DIR = Path('reports')
REPORTS_DIR.mkdir(exist_ok=True)

tf.keras.utils.set_random_seed(SEED)

for split_name in ['train', 'validation', 'test']:
    if not (DATA_DIR / split_name).is_dir():
        raise FileNotFoundError(f'Missing folder: {DATA_DIR / split_name}')

print('Clean split found:', DATA_DIR.resolve())
print('Classes:', CLASS_NAMES)
print('Protected test folder exists but will not be loaded in this notebook.')

In [ ]:
def load_split(split_name, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        DATA_DIR / split_name, labels='inferred', label_mode='binary',
        class_names=CLASS_NAMES, color_mode='rgb', image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE, shuffle=shuffle, seed=SEED if shuffle else None,
    )

train_ds = load_split('train', shuffle=True)
validation_ds = load_split('validation', shuffle=False)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.10),
], name='training_augmentation')

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda images, labels: (data_augmentation(images, training=True), labels), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
validation_ds = validation_ds.prefetch(AUTOTUNE)

images, labels = next(iter(train_ds))
print('Training batch image shape:', images.shape)
print('Training batch label shape:', labels.shape)
print('Validation receives no random augmentation.')

In [ ]:
# Naive baseline: predict Pothole for every validation image.
# It is not trained; it shows the minimum result that a real model should beat.
y_validation = np.concatenate([labels.numpy().ravel() for _, labels in validation_ds])
naive_predictions = np.ones_like(y_validation, dtype=int)
naive_scores = np.ones_like(y_validation, dtype=float)

def calculate_metrics(y_true, predictions, scores):
    return {
        'accuracy': float(accuracy_score(y_true, predictions)),
        'macro_f1': float(f1_score(y_true, predictions, average='macro', zero_division=0)),
        'pothole_precision': float(precision_score(y_true, predictions, pos_label=1, zero_division=0)),
        'pothole_recall': float(recall_score(y_true, predictions, pos_label=1, zero_division=0)),
        'normal_recall': float(recall_score(y_true, predictions, pos_label=0, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, scores)),
    }

naive_metrics = calculate_metrics(y_validation, naive_predictions, naive_scores)
display(pd.DataFrame([naive_metrics], index=['naive_majority_v1']))

In [ ]:
# First real model: a small CNN trained from scratch.
# It intentionally does NOT use class weights. Class weights are a later controlled experiment.
cnn_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=IMAGE_SIZE + (3,)),
    tf.keras.layers.Rescaling(1.0 / 255),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(1, activation='sigmoid', name='pothole_probability'),
], name='cnn_unweighted_v1')

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='roc_auc')],
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=4, restore_best_weights=True,
)
cnn_model.summary()

In [ ]:
start_time = time.time()
history = cnn_model.fit(
    train_ds, validation_data=validation_ds, epochs=20,
    callbacks=[early_stopping], verbose=1,
)
training_seconds = time.time() - start_time
print(f'Training time: {training_seconds:.1f} seconds')

In [ ]:
# Validation evaluation only. Do not replace validation_ds with the test split here.
cnn_scores = cnn_model.predict(validation_ds).ravel()
cnn_predictions = (cnn_scores >= 0.50).astype(int)
cnn_metrics = calculate_metrics(y_validation, cnn_predictions, cnn_scores)

comparison = pd.DataFrame([
    {'run_name': 'naive_majority_v1', 'hypothesis': 'Always predicting the majority class is the simplest floor.', 'changed_factor': 'naive majority rule', 'class_weight': 'not applicable', 'training_seconds': 0.0, **naive_metrics},
    {'run_name': 'cnn_unweighted_v1', 'hypothesis': 'A compact CNN can learn road-image patterns beyond the majority rule.', 'changed_factor': 'simple CNN model', 'class_weight': 'none', 'training_seconds': round(training_seconds, 1), **cnn_metrics},
])
comparison.to_csv(REPORTS_DIR / 'experiment_record.csv', index=False)
display(comparison)

print(classification_report(y_validation, cnn_predictions, target_names=CLASS_NAMES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_validation, cnn_predictions, display_labels=CLASS_NAMES)
plt.title('Validation confusion matrix: cnn_unweighted_v1')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'cnn_unweighted_v1_validation_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
history_frame = pd.DataFrame(history.history)
history_frame[['loss', 'val_loss']].plot(title='CNN training and validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'cnn_unweighted_v1_learning_curve.png', dpi=150)
plt.show()

print('Saved Colab evidence in reports/:')
for path in sorted(REPORTS_DIR.iterdir()):
    print('-', path.name)

print('Next decision: inspect validation Macro F1 and the confusion matrix before adding class weights or transfer learning.')

## What this notebook does not do

- It does not use the clean test split.
- It does not choose the final project model.
- It does not save a final deployment artifact yet.
- It does not measure pothole danger, size, severity, or repair priority.